In [1]:
import logging
import torch
import torch.nn as nn
import torch_numopt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from train_loop import train_loop

In [2]:
logging.getLogger().setLevel(logging.INFO)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [6]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardt(model=model, lr_init=1e-2, solver="cg-trunc")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.12592542171478271
epoch:  1, loss: 0.121256023645401
epoch:  2, loss: 0.11657360941171646
epoch:  3, loss: 0.11151008307933807
epoch:  4, loss: 0.1070798709988594
epoch:  5, loss: 0.1029462218284607
epoch:  6, loss: 0.09899584949016571
epoch:  7, loss: 0.09525652229785919
epoch:  8, loss: 0.09167642891407013
epoch:  9, loss: 0.08825495094060898
epoch:  10, loss: 0.0849580466747284
epoch:  11, loss: 0.08181142807006836
epoch:  12, loss: 0.07880427688360214
epoch:  13, loss: 0.0759139358997345
epoch:  14, loss: 0.07312358170747757
epoch:  15, loss: 0.07042506337165833
epoch:  16, loss: 0.06786022335290909
epoch:  17, loss: 0.06538211554288864
epoch:  18, loss: 0.06300805509090424
epoch:  19, loss: 0.060700878500938416
epoch:  20, loss: 0.05850636959075928
epoch:  21, loss: 0.056371793150901794
epoch:  22, loss: 0.054330840706825256
epoch:  23, loss: 0.05233306065201759
epoch:  24, loss: 0.05042335018515587
epoch:  25, loss: 0.0485854335129261
epoch:  26, loss: 0.046817

KeyboardInterrupt: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9457848134159312
Test metrics:  R2 = 0.7817287232818272


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardtLS(model=model, solver="cg-trunc")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.37352538108825684
epoch:  1, loss: 0.29970094561576843
epoch:  2, loss: 0.16532468795776367
epoch:  3, loss: 0.14880159497261047
epoch:  4, loss: 0.10077515244483948
epoch:  5, loss: 0.06715383380651474
epoch:  6, loss: 0.03693312034010887
epoch:  7, loss: 0.023367471992969513
epoch:  8, loss: 0.01584811508655548
epoch:  9, loss: 0.010926764458417892
epoch:  10, loss: 0.007957562804222107
epoch:  11, loss: 0.005567794665694237
epoch:  12, loss: 0.004525670316070318
epoch:  13, loss: 0.003510124748572707
epoch:  14, loss: 0.0028416032437235117
epoch:  15, loss: 0.0023688613437116146
epoch:  16, loss: 0.002029799623414874
epoch:  17, loss: 0.0017504548886790872
epoch:  18, loss: 0.001494778785854578
epoch:  19, loss: 0.0012684784596785903
epoch:  20, loss: 0.0010848639067262411
epoch:  21, loss: 0.0009250497096218169
epoch:  22, loss: 0.0007945874822326005
epoch:  23, loss: 0.0006980992620810866
epoch:  24, loss: 0.0006223334930837154
epoch:  25, loss: 0.00056784076150

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9978666273103903
Test metrics:  R2 = 0.9969504057839638
